In [ ]:
from datasets import load_dataset
dataset = load_dataset("CShorten/ML-ArXiv-Papers")

In [ ]:
import pandas as pd
df = dataset["train"].to_pandas()
df.columns

In [ ]:
df.head()

In [ ]:
df = df[['title', 'abstract']]
df = df.head(50000)
df.isnull().sum()

In [ ]:
df["paper_text"] = df["paper_text"].str.replace("\n", " ", regex=False)
df["paper_text"] = df["paper_text"].str.strip()
print(df["paper_text"].iloc[0])

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print(type(model))

In [ ]:
sample_text = df["paper_text"].iloc[0]
embedding = model.encode(sample_text)
print(type(embedding))
print(embedding.shape)   

In [ ]:
import sklearn
from sklearn.metrics.pairwise import cosine_similarity
sample_embeddings = model.encode(df["paper_text"].head(5).to_list())
print(sample_embeddings.shape) 
self_similarity = cosine_similarity(
    sample_embeddings[0].reshape(1, -1),
    sample_embeddings[0].reshape(1, -1)
)
print("Self-similarity:", self_similarity[0][0])
for i in range(1, 5):
    sim = cosine_similarity(
        sample_embeddings[0].reshape(1, -1),
        sample_embeddings[i].reshape(1, -1)
    )
    print(f"Similarity with Paper {i}: {sim[0][0]:.4f}")

In [ ]:
import numpy as np
import os
index_path = "../data/arxiv_embeddings.npy"
if os.path.exists(index_path):
    print("Loading saved embeddings...")
    embeddings = np.load(index_path)
else:
    print("Generating embeddings for all papers...")
    embeddings = model.encode(
        df["paper_text"].tolist(),
        batch_size=32,
        show_progress_bar=True
    )
    os.makedirs("../data", exist_ok=True)
    np.save(index_path, embeddings)
    print("Embeddings saved successfully!")

print(embeddings.shape)

In [ ]:
os.makedirs("../data", exist_ok=True)
df.to_csv("../data/cleaned_arxiv_papers.csv", index=False)
print("Cleaned DataFrame saved successfully!")